# Stephen V EEG model

This notebook trains a small PyTorch model on 100 generated EEG samples. The training targets rise from 0.00 to 0.99. EegML uses the Python model's scores as training targets for the separate .NET model.

**Run All** to train the Python model, save `Stephen V state.pt`, and refresh `Stephen V.csv` for EegML. You can also run the export cell on its own after loading the definitions to reuse the saved checkpoint.

## Settings

Start the kernel in the repository root or the `Yijing.py` folder. Set `FRONT_GAMMA_SCALE` to adjust the synthetic gamma ramp at AF7 and AF8. Set `MUSE_RECORDING` to choose a file for the optional recording check.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn

output_dir = Path.cwd()
if (output_dir / "Yijing.py").is_dir():
    output_dir = output_dir / "Yijing.py"

CHECKPOINT = output_dir / "Stephen V state.pt"
SYNTHETIC_CSV = output_dir / "Stephen V.csv"
MUSE_TEST_CSV = output_dir / "Stephen V test.csv"
MUSE_RECORDING = (
    Path.home() / "Documents" / "Yijing" / "Muse" / "2026-11-08-18-23-49-Muse.csv"
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAMPLE_COUNT = 100
FRONT_GAMMA_SCALE = 2.4
MUSE_TAIL_ROWS = 11_000
TRAINING_EPOCHS = 300_000
LEARNING_RATE = 0.1
TRAINING_SEED = 0

BANDS = ("Delta", "Theta", "Alpha", "Beta", "Gamma")
SENSORS = ("TP9", "AF7", "AF8", "TP10")
EEG_COLUMNS = [f"{band}_{sensor}" for band in BANDS for sensor in SENSORS]

print(f"PyTorch device: {DEVICE}")
print(f"Output folder: {output_dir}")

## Synthetic data

Each row has five frequency bands from four Muse sensors. Delta, theta, and alpha alternate between two sets of values. Beta and gamma rise with the sample number, with the largest gamma ramps at the front sensors, AF7 and AF8. Several inputs rise together, so the model can use any of them to fit the target ramp.

In [ ]:
def generate_synthetic_data():
    """Return the synthetic EEG rows and their target scores."""
    # Within each band, values are ordered TP9, AF7, AF8, TP10.
    even_baseline = [
        0.2, 0.0, 0.0, 0.2,  # Delta
        0.2, 0.0, 0.0, 0.2,  # Theta
        0.8, 0.5, 0.5, 0.8,  # Alpha
    ]
    odd_baseline = [
        -0.1, -0.5, -0.5, -0.1,  # Delta
        -0.1, -0.5, -0.5, -0.1,  # Theta
        0.4, 0.0, 0.0, 0.4,  # Alpha
    ]
    ramp_scales = [
        0.5, 1.4, 1.4, 0.5,  # Beta
        0.4, FRONT_GAMMA_SCALE, FRONT_GAMMA_SCALE, 0.4,  # Gamma
    ]

    rows = []
    targets = []
    for sample in range(SAMPLE_COUNT):
        baseline = even_baseline if sample % 2 == 0 else odd_baseline
        ramp = [scale * sample / SAMPLE_COUNT for scale in ramp_scales]
        rows.append(baseline + ramp)
        targets.append([sample / SAMPLE_COUNT])

    return (
        torch.tensor(rows, dtype=torch.float32, device=DEVICE),
        torch.tensor(targets, dtype=torch.float32, device=DEVICE),
    )

## Python model and training

The model turns a row of 20 EEG values into a score between 0 and 1. Training uses a fixed seed so the weights start from the same values on each run with the same setup. With these fractional targets, even an exact fit has a binary cross-entropy loss of about 0.4999.

In [ ]:
class EegNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(len(EEG_COLUMNS), 100),
            nn.ReLU(),
            nn.Linear(100, 1),
            nn.Sigmoid(),
        )

    def forward(self, inputs):
        return self.model(inputs)


def train_eeg_model1():
    """Train on the generated rows and save the model weights."""
    inputs, targets = generate_synthetic_data()
    torch.manual_seed(TRAINING_SEED)
    model = EegNN().to(DEVICE)
    loss_function = nn.BCELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)

    for epoch in range(1, TRAINING_EPOCHS + 1):
        optimizer.zero_grad()
        loss = loss_function(model(inputs), targets)
        loss.backward()
        optimizer.step()

        if epoch % 100_000 == 0 or epoch == TRAINING_EPOCHS:
            print(f"Epoch {epoch:,}/{TRAINING_EPOCHS:,} | Loss {loss.item():.4f}")

    torch.save(model.state_dict(), CHECKPOINT)
    print(f"Saved model weights to {CHECKPOINT}")

## Score and export

`test_eeg_model1()` scores the generated rows and refreshes the EegML training CSV. The synthetic target line in the plot lets you check how well Python fitted those same rows. The CSV stores the model's score in `Prediction`, followed by all 20 EEG columns, starting with `Delta_TP9`.

Pass a Muse filename to score up to 100 rows from the start of the last 11,000 rows in that recording. That mode writes to the separate test CSV.

In [ ]:
def test_eeg_model1(recording=None):
    """Use the saved model to score synthetic rows or a Muse CSV."""
    if recording is None:
        inputs, targets = generate_synthetic_data()
        output_file = SYNTHETIC_CSV
        title = "Synthetic EEG"
    else:
        data = pd.read_csv(recording, usecols=EEG_COLUMNS)
        data = data.tail(MUSE_TAIL_ROWS).head(SAMPLE_COUNT)
        values = data[EEG_COLUMNS].to_numpy(dtype="float32", copy=True)
        inputs = torch.from_numpy(values).to(DEVICE)
        targets = None
        output_file = MUSE_TEST_CSV
        title = "Muse recording"

    model = EegNN().to(DEVICE)
    model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE, weights_only=True))
    model.eval()
    with torch.inference_mode():
        predictions = model(inputs).squeeze(1).cpu().numpy()

    results = pd.DataFrame(inputs.cpu().numpy(), columns=EEG_COLUMNS)
    results.insert(0, "Prediction", predictions)
    results.to_csv(output_file, index=False)
    print(f"Wrote {len(results)} rows to {output_file}")

    figure, (score_plot, gamma_plot) = plt.subplots(
        2, 1, figsize=(10, 6), sharex=True, layout="constrained"
    )
    figure.suptitle(title)
    score_plot.plot(results["Prediction"], label="Prediction")
    if targets is not None:
        score_plot.plot(targets.squeeze(1).cpu().numpy(), "--", label="Synthetic target")
    score_plot.set(ylabel="Score", ylim=(0, 1))
    score_plot.legend()

    results[["Gamma_AF7", "Gamma_AF8"]].plot(ax=gamma_plot)
    gamma_plot.set(xlabel="Sample", ylabel="Front gamma")
    plt.show()

    return results

## Run the synthetic workflow

Run the training cell to fit a fresh model, then run the export cell to write the CSV. Once the checkpoint exists, you can rerun just the export cell.

In [ ]:
train_eeg_model1()

In [ ]:
synthetic_scores = test_eeg_model1()

## Optional Muse recording check

Uncomment and run the cell below to score the recording selected in Settings. This writes `Stephen V test.csv` for inspection.

In [ ]:
muse_scores = test_eeg_model1(MUSE_RECORDING)